<a href="https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task (type)

**Type: Scoring**

I chose scoring because my goal is to measure how urgently each page needs a refresh. Unlike classification or clustering, refresh priority is a continuous value, not a category. Scoring assigns each page an interpretable number that content teams can sort, threshold, and track over time, making it the most practical choice for content refresh prioritization.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

**Proxy: a composite refresh-urgency score**, built from `trend_pct` (recent traffic momentum) and `days_since_last_update` (content staleness).

There's no single "refresh urgency" column in the raw data, so I derive a proxy: pages losing traffic fast (very negative `trend_pct`) and going longer without an update (`days_since_last_update`) are proxies for "needs attention soon." This proxy is directional and observed from behavior, not a ground-truth label — it approximates urgency using signals the business already tracks.

In [14]:
# Simple proxy: normalize trend decline and staleness, combine into one score
df['trend_penalty'] = -df['trend_pct'].clip(upper=0)  # bigger drop = bigger penalty
df['staleness_penalty'] = df['days_since_last_update']

# Normalize both to 0-1 range, then average
df['trend_norm'] = df['trend_penalty'] / df['trend_penalty'].max()
df['staleness_norm'] = df['staleness_penalty'] / df['staleness_penalty'].max()

df['refresh_urgency_score'] = (df['trend_norm'] + df['staleness_norm']) / 2

df[['content_id', 'trend_pct', 'days_since_last_update', 'refresh_urgency_score']].head()


,content_id,trend_pct,days_since_last_update,refresh_urgency_score
0,content_304f48230142,-41.4,20,0.233810
1,content_a1fb4e703a9e,-57.7,25,0.322012
2,content_9aa793d4d895,-60.9,20,0.331310
3,content_331d6c4de07b,-13.8,22,0.098491
4,content_d99b7a2d90ca,-34.7,14,0.192267


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Mean Absolute Error (MAE)

I chose MAE because it directly measures the average prediction error, making it easy to interpret. A good result is a low MAE. For example, if the target is measured in days, an MAE below 7 days means predictions are typically within a week of the true refresh urgency, making the scores useful for content prioritization.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one piece of content (`content_id`), belonging to one client (`client_id`). Each row captures that page's freshness signals (`days_since_last_update`, `freshness_tier`), performance trend (`trend_direction`, `trend_pct`), and engagement behavior (`engagement_rate`, `ctr`, `avg_position`) — the exact inputs needed to score how urgently that page needs a refresh.

In [16]:

unit_df = df[['content_id', 'client_id', 'days_since_last_update', 'freshness_tier',
              'trend_direction', 'trend_pct', 'engagement_rate', 'ctr', 'avg_position']]
unit_df.head()

,content_id,client_id,days_since_last_update,freshness_tier,trend_direction,trend_pct,engagement_rate,ctr,avg_position
0,content_304f48230142,client_f369cb89fc,20,0-30,down,-41.4,5.88,0.76,10.6
1,content_a1fb4e703a9e,client_4e07408562,25,0-30,down,-57.7,0.00,0.05,20.3
2,content_9aa793d4d895,client_7f2253d7e2,20,0-30,down,-60.9,0.00,0.09,36.5
3,content_331d6c4de07b,client_19581e27de,22,0-30,stable,-13.8,1.28,0.49,6.2
4,content_d99b7a2d90ca,client_3fdba35f04,14,0-30,down,-34.7,0.00,0.13,44.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The pattern is too complex for a simple if-statement because refresh urgency depends on **multiple signals working together**, not one fixed rule. The same traffic drop can have different meanings depending on factors like content age, update history, and page type. A machine learning model can learn these relationships from data, producing more accurate and adaptable refresh scores than static thresholds.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.